# BlockScalar_Layer — Gradient & DeltaRule 수치 검증

1. **Gradient test**: `gradient(bs, ...)` 가 finite difference 와 일치하는지 확인 (W 에 대한 편미분)
2. **DeltaRule test**: `DeltaRule(bs, ...)` 가 finite difference 와 일치하는지 확인 (입력 fermion 에 대한 편미분)

In [1]:
using LatticeDiracOperators
using Gaugefields
using Wilsonloop
using LMP
using LinearAlgebra

┌ Warning: Circular dependency detected. Precompilation will be skipped for:
│   ChainRules [082447d4-558c-5d27-93f4-14fc19e9eca2]
│   SpecialFunctionsExt [1285c0f1-ff9f-5867-b66e-0f359bcf09ba]
│   GPUArrays [0c68f7d7-f131-5f86-a1c3-88cf8149b2d7]
│   LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018]
│   KernelAbstractions [63c18a36-062a-441e-b654-da1e3ab1ce7c]
│   CUDA [052768ef-5323-5732-b1bb-66c8b64840ba]
│   SparseArraysExt [85068d23-b5fb-53f1-8204-05c2aba6942f]
│   StructArraysGPUArraysCoreExt [cbbf52aa-d660-55d4-8a5a-33845bbaa85d]
│   AtomixCUDAExt [13011619-4c7c-5ef0-948f-5fc81565cd05]
│   LinearAlgebraExt [66d79d19-2cc4-5b0b-ac7a-b340256d1ecd]
│   ZygoteColorsExt [e68c091a-8ea5-5ca7-be4f-380657d4ad79]
│   ChainRulesCoreExt [eae2faf6-b232-58cb-a410-7764fda2830c]
│   Zygote [e88e6eb3-aa80-5325-afca-941959d7151f]
└ @ Pkg.API C:\Users\_s2220186\AppData\Local\Programs\Julia-1.10.7\share\julia\stdlib\v1.10\Pkg\src\API.jl:1279
[ Info: Precompiling LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018]
[ 

In [2]:
# ── 격자 설정 (CPU, 4^4) ──
NX = 4; NY = 4; NZ = 4; NT = 4
Nwing = 0; Dim = 4; NC = 3

U = Initialize_Gaugefields(NC, Nwing, NX, NY, NZ, NT, condition="hot")

x = Initialize_pseudofermion_fields(U[1], "Wilson", nowing=true)

params = Dict()
params["Dirac_operator"] = "Wilson"
params["κ"] = 0.16
params["faster version"] = true
params["eps_CG"] = 1.0e-8
params["verbose_level"] = 2

D = Dirac_operator(U, x, params)
println("Lattice: $(NX)×$(NY)×$(NZ)×$(NT), NC=$NC")

Lattice: 4×4×4×4, NC=3


In [3]:
# ── 레이어 구성: [fe, bs, Ritz] ──
n_vec = 2    # 벡터 수
r     = 2    # block 크기

fe   = fermionfield_Layer(x, n_vec)
bs   = BlockScalar_Layer(x, n_vec, r)

# W를 복소수로 랜덤 초기화 (실수와 허수 양쪽 gradient를 동시에 검증)
bs.W .= randn(ComplexF64, size(bs.W))

Ritz = Ritz_Layer(D)

layers =[fe, bs, Ritz]

nblocks = size(bs.W, 1)
println("nblocks = $nblocks, n_vec = $n_vec, r = $r")
println("W size = $(size(bs.W))")

nblocks = 16, n_vec = 2, r = 2
W size = (16, 2)


## 1. Gradient 수치 검증 — Simple Loss (Ritz 없이)

BlockScalar 단독으로 gradient 공식이 맞는지 확인.
L = Σ_k norm2(bs(x)[k]) (심플한 squared norm 손실)

In [ ]:
# ── Simple loss: L = Σ_k norm2(y_k) ──
y = bs(fe.x)
L_simple = sum(real(norm2(yk)) for yk in y)

# delta = 2*y  (∂||y||²/∂y^* = y, framework convention: factor 2 포함)
delta_simple = [2 * yk for yk in y]

# gradient 함수로 analytical gradient 계산
γ = Clifford()
grad_simple_ana = gradient(bs, bs.activation, fe.x, delta_simple, γ)

println("Simple Loss L = $L_simple")
println("Sample: grad_simple_ana[1,1] = $(grad_simple_ana[1,1])")

Loss f = 2.8080668603687773


In [ ]:
# ── Numerical gradient for simple loss (central difference) ──
epsilon = 1e-5
W_orig = copy(bs.W)
grad_simple_num = zeros(ComplexF64, nblocks, n_vec)

for k in 1:n_vec
    for i in 1:nblocks
        bs.W = copy(W_orig); bs.W[i, k] += epsilon
        yp = bs(fe.x); Lp = sum(real(norm2(yk)) for yk in yp)
        bs.W = copy(W_orig); bs.W[i, k] -= epsilon
        ym = bs(fe.x); Lm = sum(real(norm2(yk)) for yk in ym)

        bs.W = copy(W_orig); bs.W[i, k] += im * epsilon
        yp = bs(fe.x); Lp_im = sum(real(norm2(yk)) for yk in yp)
        bs.W = copy(W_orig); bs.W[i, k] -= im * epsilon
        ym = bs(fe.x); Lm_im = sum(real(norm2(yk)) for yk in ym)

        grad_simple_num[i, k] = complex((Lp - Lm) / (2epsilon), (Lp_im - Lm_im) / (2epsilon))
    end
end
bs.W = copy(W_orig)

max_err_simple = maximum(abs.(grad_simple_ana .- grad_simple_num))
println("=== Simple Loss Gradient Test ===")
println("max |analytical - numerical| = $max_err_simple")
if max_err_simple < 1e-5
    println("✓ PASS")
else
    println("✗ FAIL")
end

Analytical gradient size = (16, 2)
Sample: grad_ana[1,1] = -0.0019987587219044266 + 0.0002315699021775764im


## 2. Gradient 수치 검증 — Ritz Loss

forward_propagation / backward_propagation 을 사용한 전체 파이프라인 테스트.

In [ ]:
# ── Ritz loss: forward / backward propagation ──
u, f = forward_propagation(layers, D, D)
delta = backward_propagation(layers, u, 1)
grad_ana = gradient(bs, bs.activation, u[2], delta[1], γ)

println("Ritz Loss f = ", real(f))
println("Sample: grad_ana[1,1] = $(grad_ana[1,1])")

In [6]:
# ── Numerical gradient (central difference) ──
epsilon = 1e-5
W_orig = copy(bs.W)
grad_num = zeros(ComplexF64, nblocks, n_vec)

for k in 1:n_vec
    for i in 1:nblocks
        # ∂L/∂(Re W[i,k])  — central difference
        bs.W = copy(W_orig)
        bs.W[i, k] += epsilon
        _, f_plus = forward_propagation(layers, D, D)
        bs.W = copy(W_orig)
        bs.W[i, k] -= epsilon
        _, f_minus = forward_propagation(layers, D, D)
        dL_dReW = (real(f_plus) - real(f_minus)) / (2 * epsilon)

        # ∂L/∂(Im W[i,k])  — central difference
        bs.W = copy(W_orig)
        bs.W[i, k] += im * epsilon
        _, f_plus = forward_propagation(layers, D, D)
        bs.W = copy(W_orig)
        bs.W[i, k] -= im * epsilon
        _, f_minus = forward_propagation(layers, D, D)
        dL_dImW = (real(f_plus) - real(f_minus)) / (2 * epsilon)

        grad_num[i, k] = complex(dL_dReW, dL_dImW)
    end
end
bs.W = copy(W_orig)  # 원래 값 복원

println("Numerical gradient done (central difference, ε=$epsilon).")
println("Sample: grad_num[1,1] = $(grad_num[1,1])")

Numerical gradient done (central difference, ε=1.0e-5).
Sample: grad_num[1,1] = -0.0010475717626690084 + 0.0048283491427270064im


In [7]:
# ── Gradient 비교 ──
diff = grad_ana .- grad_num
max_err = maximum(abs.(diff))
println("=== Gradient Test (grad_ana vs grad_num) ===")
println("max |grad_ana - grad_num| = $max_err")
if max_err < 1e-4
    println("✓ PASS")
else
    println("✗ FAIL — 아래 대안 비교 확인")
    # 대안: 혹시 grad_num / 2 또는 conj(grad_num) 과 일치하는지
    println("max |grad_ana - grad_num/2|       = $(maximum(abs.(grad_ana .- grad_num./2)))")
    println("max |grad_ana - conj(grad_num)|   = $(maximum(abs.(grad_ana .- conj.(grad_num))))")
    println("max |grad_ana - 2*grad_num|       = $(maximum(abs.(grad_ana .- 2 .* grad_num)))")
    println("max |grad_ana - 2*conj(grad_num)| = $(maximum(abs.(grad_ana .- 2 .* conj.(grad_num))))")
end

=== Gradient Test (grad_ana vs grad_num) ===
max |grad_ana - grad_num| = 0.02653485701584465
✗ FAIL — 아래 대안 비교 확인
max |grad_ana - grad_num/2|       = 0.02045633487655573
max |grad_ana - conj(grad_num)|   = 0.026510092085117367
max |grad_ana - 2*grad_num|       = 0.03869373002292774
max |grad_ana - 2*conj(grad_num)| = 0.03865976505499463


## 3. DeltaRule 수치 검증

BlockScalar 의 DeltaRule 으로 전파된 delta 가 올바른지:
입력(fe.x 의 개별 성분)을 섭동 → 손실 변화로 수치 미분을 구해 비교.

In [8]:
# ── DeltaRule output (bs 를 통해 전파된 delta) ──
# backward_propagation 비-! 버전에서:
#   delta[2] = DeltaRule(bs, ..., u[2], delta[1])  ← fe 에 전달되는 delta
# gradient(fe, ...) = delta[2] 그대로 반환.
delta_fe = delta[2]   # Vector{WF3x1}, length = n_vec

# 검증할 성분 인덱스 (k=벡터, ic=color, ix-it=좌표, ig=spinor)
NC_val = fe.x[1].NC
NX_val, NY_val, NZ_val, NT_val = fe.x[1].NX, fe.x[1].NY, fe.x[1].NZ, fe.x[1].NT
NG_val = fe.x[1].NG

# 무작위로 10 개 성분 선택
using Random
Random.seed!(42)
n_test = 10
test_pts = [(rand(1:n_vec),
             rand(1:NC_val),
             rand(1:NX_val), rand(1:NY_val), rand(1:NZ_val), rand(1:NT_val),
             rand(1:NG_val)) for _ in 1:n_test]

println("DeltaRule 검증: $(n_test) 개 성분 테스트")

DeltaRule 검증: 10 개 성분 테스트


In [9]:
# ── Numerical delta (입력 fe.x 의 개별 성분 섭동) ──
epsilon_d = 1e-7

println("idx | numerical (re, im) | analytical (delta) | diff")
println("-"^80)

max_delta_err = 0.0

for (idx, (k, ic, ix, iy, iz, it, ig)) in enumerate(test_pts)
    orig_val = fe.x[k][ic, ix, iy, iz, it, ig]

    # ∂L/∂(Re x)
    fe.x[k][ic, ix, iy, iz, it, ig] = orig_val + epsilon_d
    _, f_re = forward_propagation(layers, D, D)
    dL_dRe = (real(f_re) - real(f)) / epsilon_d

    # ∂L/∂(Im x)
    fe.x[k][ic, ix, iy, iz, it, ig] = orig_val + im * epsilon_d
    _, f_im = forward_propagation(layers, D, D)
    dL_dIm = (real(f_im) - real(f)) / epsilon_d

    # 복원
    fe.x[k][ic, ix, iy, iz, it, ig] = orig_val

    num_grad = complex(dL_dRe, dL_dIm)
    ana_val  = delta_fe[k][ic, ix, iy, iz, it, ig]

    d = abs(ana_val - num_grad)
    global max_delta_err = max(max_delta_err, d)

    println("$idx | num=$(round(num_grad, sigdigits=6)) | ana=$(round(ana_val, sigdigits=6)) | diff=$(round(d, sigdigits=3))")
end

println()
println("max |delta_analytical - numerical| = $max_delta_err")

idx | numerical (re, im) | analytical (delta) | diff
--------------------------------------------------------------------------------
1 | num=-0.000162106 + 0.000217382im | ana=-0.000232363 - 0.00013978im | diff=0.000364
2 | num=2.37232e-5 + 2.29816e-5im | ana=1.32623e-5 - 3.02506e-5im | diff=5.43e-5
3 | num=-0.000224585 + 0.000340594im | ana=0.000244842 - 0.000326338im | diff=0.000816
4 | num=-8.07798e-5 - 0.000429461im | ana=0.000367154 + 0.000236961im | diff=0.000803
5 | num=-0.000178142 + 0.000146754im | ana=0.000170267 + 0.000155818im | diff=0.000349
6 | num=-0.00016545 - 0.000214788im | ana=6.46525e-5 + 0.000263301im | diff=0.000531
7 | num=-0.000177747 + 0.000107097im | ana=6.02426e-5 - 0.000198578im | diff=0.000387
8 | num=0.000357403 + 6.14797e-5im | ana=0.000307485 - 0.000192278im | diff=0.000259
9 | num=-0.000115805 - 1.40643e-5im | ana=-0.000116588 - 4.19016e-6im | diff=9.91e-6
10 | num=-7.18492e-5 + 0.000585936im | ana=-0.000569688 - 0.000154727im | diff=0.000892

max |del

In [10]:
# ── DeltaRule 판정 ──
if max_delta_err < 1e-4
    println("✓ DeltaRule PASS")
else
    println("✗ DeltaRule FAIL — 아래 대안 비교 확인")
    # 첫 번째 테스트 성분으로 대안 관계 탐색
    k, ic, ix, iy, iz, it, ig = test_pts[1]
    orig_val = fe.x[k][ic, ix, iy, iz, it, ig]

    fe.x[k][ic, ix, iy, iz, it, ig] = orig_val + epsilon_d
    _, f_re2 = forward_propagation(layers, D, D)
    fe.x[k][ic, ix, iy, iz, it, ig] = orig_val + im * epsilon_d
    _, f_im2 = forward_propagation(layers, D, D)
    fe.x[k][ic, ix, iy, iz, it, ig] = orig_val

    num = complex((real(f_re2)-real(f))/epsilon_d, (real(f_im2)-real(f))/epsilon_d)
    ana = delta_fe[k][ic, ix, iy, iz, it, ig]

    println("num              = $num")
    println("ana              = $ana")
    println("conj(ana)        = $(conj(ana))")
    println("ana/2            = $(ana/2)")
    println("conj(ana)/2      = $(conj(ana)/2)")
    println("2*ana            = $(2*ana)")
    println("2*conj(ana)      = $(2*conj(ana))")
    println("complex(Re,-Im)  = $(complex(real(ana),-imag(ana)))")
end

✗ DeltaRule FAIL — 아래 대안 비교 확인
num              = -0.00016210588427156836 + 0.00021738166822160565im
ana              = -0.0002323629285682684 - 0.00013977978753884323im
conj(ana)        = -0.0002323629285682684 + 0.00013977978753884323im
ana/2            = -0.0001161814642841342 - 6.988989376942161e-5im
conj(ana)/2      = -0.0001161814642841342 + 6.988989376942161e-5im
2*ana            = -0.0004647258571365368 - 0.00027955957507768645im
2*conj(ana)      = -0.0004647258571365368 + 0.00027955957507768645im
complex(Re,-Im)  = -0.0002323629285682684 + 0.00013977978753884323im
